In [1]:
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import recall_score, f1_score, confusion_matrix

df = pd.read_csv(r'C:\Users\bachu\Desktop\5thsem_OJT\project1_diabetes\data\processed\muhammed_bilal_p1w2_features.csv')


feature_cols = ["age","pregnancies","glucose","blood_pressure","skin_thickness","insulin","bmi","diabetes_pedigree","bmi_category","glucose_band"]
x= pd.get_dummies(df[feature_cols],drop_first=True)
y=df["outcome"]

x_train,x_test,y_train,y_test= train_test_split(x,y,test_size=0.2,random_state=42,stratify=y)

In [2]:
candidates = {
    "LogReg": (
        "scaled",
        LogisticRegression(max_iter=1000)
    ),
    "Decision Tree": (
        "raw",
        DecisionTreeClassifier(max_depth=3, random_state=42)
    ),
    "KNN": (
        "scaled",
        KNeighborsClassifier(n_neighbors=25)
    ),
    "SVC balanced": (
        "scaled",
        SVC(class_weight="balanced", random_state=42)
    ),
    "RF balanced": (
        "raw",
        RandomForestClassifier(
            max_depth=5,
            class_weight="balanced",
            random_state=42
        )
    )
}

TARGET_RECALL, GUARD_F1 = 0.70, 0.50  # Week 5 decision

In [3]:
print ('rows:',len(df),"| train:",len(x_train),"| test:", len(x_test))
print("diabetic in test set:", int(y_test.sum()))
print("featres after get dummies:",x.shape[1])
print("candidates:", list(candidates))
print("target: recall >=", TARGET_RECALL, "with F1 >=", GUARD_F1)

rows: 950 | train: 760 | test: 190
diabetic in test set: 55
featres after get dummies: 13
candidates: ['LogReg', 'Decision Tree', 'KNN', 'SVC balanced', 'RF balanced']
target: recall >= 0.7 with F1 >= 0.5


In [4]:
rows = []

for rs in [0, 1, 7, 42, 99]:
    Xa, Xb, ya, yb = train_test_split(
        x, y,
        test_size=0.2,
        random_state=rs,
        stratify=y
    )

    row = {"split": rs}

    for name, (kind, model) in candidates.items():
        m = clone(model)  

        if kind == "scaled":
            sc = StandardScaler().fit(Xa)
            m.fit(sc.transform(Xa), ya)
            pred = m.predict(sc.transform(Xb))

        else:
            m.fit(Xa, ya)
            pred = m.predict(Xb)
        row[name]=round(recall_score(yb,pred),3)
    rows.append(row)  

splits=pd.DataFrame(rows).set_index('split')
print('splits')  
spread=(splits.max() - splits.min()).round(3)
print('\nspread (best - worst):\n', spread.sort_values(ascending=False))

splits

spread (best - worst):
 Decision Tree    0.291
SVC balanced     0.145
LogReg           0.073
KNN              0.073
RF balanced      0.073
dtype: float64


# CROSS VALIDATION


K-Fold Cross-Validation is a technique used to check how well a machine learning model performs on different parts of a dataset.

In [5]:
from sklearn.model_selection import StratifiedKFold

cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for i,(tr,te) in enumerate(cv.split(x_train,y_train),1):
    print(f'fold{i}: train {len(tr)} score{len(te)}'
          f'diabetic in score fold{y_train.iloc[te].sum()}')

def make_model(kind, model):
    """Scaled models get the scaler INSIDE a Pipeline,so it refits per fold."""
    if kind == 'scaled':
        return Pipeline([("scale",StandardScaler()),('model',model)])
    return model

print()
for name,(kind,model)in candidates.items():
    print(f"{name:15s}->{type(make_model(kind,model)).__name__}")

fold1: train 608 score152diabetic in score fold43
fold2: train 608 score152diabetic in score fold43
fold3: train 608 score152diabetic in score fold44
fold4: train 608 score152diabetic in score fold44
fold5: train 608 score152diabetic in score fold44

LogReg         ->Pipeline
Decision Tree  ->DecisionTreeClassifier
KNN            ->Pipeline
SVC balanced   ->Pipeline
RF balanced    ->RandomForestClassifier


Setup cros validation the right way

Cross validate every candidate

In [6]:
from sklearn.model_selection import cross_validate

rows=[]
for name, (kind,model) in candidates.items():
    est=make_model(kind,model)
    res=cross_validate(est, x_train,y_train,cv=cv,
                       scoring=['recall','f1'])
    single=clone(est).fit(x_train,y_train)
    rows.append({
        'model':    name,
        'cv_recall': round(res["test_recall"].mean(),3),
        'cv_std':  round(res["test_recall"].std(),3),
        'cv_f1':   round(res["test_f1"].mean(),3),
        'week6_recall': round(recall_score(y_test,single.predict(x_test)),3),
        'folds':  np.round(res['test_recall'],3).tolist(),
    })

cv_table=pd.DataFrame(rows).sort_values('cv_recall',ascending=False)
print(cv_table.to_string(index=False))

        model  cv_recall  cv_std  cv_f1  week6_recall                               folds
 SVC balanced      0.721   0.046  0.559         0.618 [0.744, 0.791, 0.682, 0.727, 0.659]
  RF balanced      0.656   0.054  0.523         0.655 [0.721, 0.651, 0.705, 0.636, 0.568]
       LogReg      0.303   0.073  0.390         0.291  [0.419, 0.326, 0.318, 0.25, 0.205]
Decision Tree      0.216   0.076  0.296         0.291 [0.116, 0.326, 0.159, 0.205, 0.273]
          KNN      0.174   0.018  0.261         0.182 [0.186, 0.163, 0.159, 0.205, 0.159]


In [7]:
cv_table['low'] = (cv_table['cv_recall'] - cv_table['cv_std']).round(3)
cv_table['high'] = (cv_table['cv_recall']+ cv_table['cv_std']).round(3)
cv_table['clears_target'] = cv_table['cv_recall'] >= TARGET_RECALL
cv_table['clear_guard'] = cv_table['cv_f1'] >= GUARD_F1

print(cv_table[['model','cv_recall','low','high','cv_f1',
                'clears_target','clear_guard']].to_string(index=False))

        model  cv_recall   low  high  cv_f1  clears_target  clear_guard
 SVC balanced      0.721 0.675 0.767  0.559           True         True
  RF balanced      0.656 0.602 0.710  0.523          False         True
       LogReg      0.303 0.230 0.376  0.390          False        False
Decision Tree      0.216 0.140 0.292  0.296          False        False
          KNN      0.174 0.156 0.192  0.261          False        False


HYPERPARAMETER

Hyperparameter Tuning
Simple definition: Hyperparameter tuning is the process of finding the best settings for a machine learning model to improve its performance.

Example: Finding the best value of K in the KNN algorithm.


GridSearchCV


Simple definition: GridSearchCV tests all the specified combinations of hyperparameter values and selects the best combination using cross-validation.

Example: Testing KNN with K = 3, 5, 7, and 9 to find the best K value.

RandomizedSearchCV


Simple definition: RandomizedSearchCV tests a randomly selected number of hyperparameter combinations to find a good combination efficiently.

Example: From many possible KNN settings, it randomly selects and tests a limited number of combinations

1. Hyperparameter: In KNN, we choose \(K = 5\) before making predictions. The value of K controls how many nearest neighbours are considered.

2. Normal parameter: In linear regression, the model learns the weight (coefficient) from the training data.

In [8]:
rf = RandomForestClassifier(random_state=42)
for setting,value in rf.get_params().items():
    print(f'{setting:25s}{value}')

bootstrap                True
ccp_alpha                0.0
class_weight             None
criterion                gini
max_depth                None
max_features             sqrt
max_leaf_nodes           None
max_samples              None
min_impurity_decrease    0.0
min_samples_leaf         1
min_samples_split        2
min_weight_fraction_leaf 0.0
monotonic_cst            None
n_estimators             100
n_jobs                   None
oob_score                False
random_state             42
verbose                  0
warm_start               False


buid the grid and the cost

In [9]:
param_grid ={
    "max_depth"          : [ 3,5,8,None],
    "min_samples_leaf": [1,5,10],
    "class_weight" : ["balanced","balanced_subsample"]
    }

combos = 1

for values in param_grid.values():
    combos *= len(values)

folds = cv.get_n_splits()
total_fits = combos*folds

print("combinations:", combos)
print("cross-validation folds:",folds)
print("Total model fits:",total_fits)

combinations: 24
cross-validation folds: 5
Total model fits: 120


run the grid search on the leading forest

In [10]:
from sklearn.model_selection import GridSearchCV
#create the grid search
gs_recall = GridSearchCV(
    RandomForestClassifier(random_state=42),
param_grid,
cv=cv,
scoring='recall',
n_jobs=-1)
gs_recall.fit(x_train,y_train)
print('best settings:',gs_recall.best_params_)
print('best cv recall:',round(gs_recall.best_score_,3))

best settings: {'class_weight': 'balanced', 'max_depth': 3, 'min_samples_leaf': 1}
best cv recall: 0.739


In [11]:
print("Best settings:",gs_recall.best_params_)
print("best CV recall:", round(gs_recall.best_score_,3))

top= pd.DataFrame(gs_recall.cv_results_)

print(
    top[["params","mean_test_score","std_test_score","rank_test_score"]]
    .sort_values("rank_test_score")
    .head(5)
    .to_string(index=False)
)

Best settings: {'class_weight': 'balanced', 'max_depth': 3, 'min_samples_leaf': 1}
best CV recall: 0.739
                                                                        params  mean_test_score  std_test_score  rank_test_score
           {'class_weight': 'balanced', 'max_depth': 3, 'min_samples_leaf': 1}         0.738689        0.041181                1
           {'class_weight': 'balanced', 'max_depth': 3, 'min_samples_leaf': 5}         0.738584        0.041746                2
          {'class_weight': 'balanced', 'max_depth': 3, 'min_samples_leaf': 10}         0.734249        0.062548                3
{'class_weight': 'balanced_subsample', 'max_depth': 3, 'min_samples_leaf': 10}         0.720719        0.064830                4
 {'class_weight': 'balanced_subsample', 'max_depth': 3, 'min_samples_leaf': 5}         0.711416        0.064128                5


test the tuned model on unseen patients

In [12]:
best_rf = gs_recall.best_estimator_
pred = best_rf.predict(x_test)
tn,fp,fn,tp = confusion_matrix(y_test,pred).ravel()
print('search CV recall:', round (gs_recall.best_score_,3))
print('test recall:',round(recall_score(y_test,pred),3))
print('test f1:', round (f1_score(y_test,pred),3))
print(f'diabetic patient found;{tp} of {tp+fn}')
print(f'missed patients :{fn}')
print(f'false alarms: {fp}')      

search CV recall: 0.739
test recall: 0.745
test f1: 0.536
diabetic patient found;41 of 55
missed patients :14
false alarms: 57


change the scoring,change the winner

In [13]:
from sklearn.model_selection import GridSearchCV

# Create the grid search
gs_f1 = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=cv,
    scoring='f1',
    n_jobs=-1,
    
    )

# Run the search 
gs_f1.fit(x_train, y_train)




#compare both tuned models on unseen test patients 
for label, gs in [
    ('Searched on recall', gs_recall),
    ('Searched on F1', gs_f1)    
]:
    pred = gs.best_estimator_.predict(x_test)

    found = int(((pred == 1) & (y_test.values == 1)).sum())

    print(label, ':', gs.best_params_)
    print('Test recall    :', round(recall_score(y_test, pred),3))
    print('Test F1        :', round(f1_score(y_test, pred),3))
    print('Patients found :', found)

Searched on recall : {'class_weight': 'balanced', 'max_depth': 3, 'min_samples_leaf': 1}
Test recall    : 0.745
Test F1        : 0.536
Patients found : 41
Searched on F1 : {'class_weight': 'balanced', 'max_depth': 5, 'min_samples_leaf': 10}
Test recall    : 0.655
Test F1        : 0.522
Patients found : 36


tune the runner-up with a random search

In [14]:
from sklearn.model_selection import RandomizedSearchCV

# Create SVM pipeline 

svc_pipe = Pipeline([
    ('Scale', StandardScaler()),
    ('model', SVC(class_weight='balanced', random_state=42))
])

# Hyperparameter combination
svc_grid = {
    'model__C'    : [0.1, 0.5, 1, 5, 10, 50],
    'model__gamma' : ['scale', 0.005, 0.01, 0.05, 0.1]
}

# Tune using 15 random combinations
rs_svc = RandomizedSearchCV(
    svc_pipe, svc_grid,
    n_iter=15, cv=cv,
    scoring='recall',
    random_state=42, n_jobs=-1
)

rs_svc.fit(x_train, y_train)

# Test the best SVM 
svc_pred = rs_svc.best_estimator_.predict(x_test)

svc_recall = recall_score(y_test , svc_pred)
svc_f1 = f1_score(y_test, svc_pred)

print("Best settings   :", rs_svc.best_params_)
print('CV recall       :', round(rs_svc.best_score_,3))
print("Test recall     :", round(svc_recall,3))
print('Test f1         :', round(svc_f1,3))
print('Optimism gap    :', round(rs_svc.best_score_ - svc_recall,3))
print('Flagged per 100 :', round(svc_pred.mean() * 100,1))

# compare with tuned Random forest 
forest_recall = recall_score(y_test,pred)
forest_gap = gs_recall.best_score_ - forest_recall

print('forest optimism gap:', round(forest_gap,3))

Best settings   : {'model__gamma': 'scale', 'model__C': 0.1}
CV recall       : 0.766
Test recall     : 0.673
Test f1         : 0.507
Optimism gap    : 0.094
Flagged per 100 : 47.9
forest optimism gap: 0.084


build the final ranking table

In [15]:
def evaluate(name, est):
    res = cross_validate(
        est, x_train, y_train,
        cv=cv,
        scoring=["recall", "f1"]
    )
    p = clone(est).fit(x_train, y_train).predict(x_test)
    tn, fp, fn, tp = confusion_matrix(y_test, p).ravel()
    return {
        "model": name,
        "cv_recall": round(res["test_recall"].mean(), 3),
        "cv_std": round(res["test_recall"].std(), 3),
        "cv_f1": round(res["test_f1"].mean(), 3),
        "test_recall": round(recall_score(y_test, p), 3),
        "test_f1": round(f1_score(y_test, p), 3),
        "found": tp,
        "missed": fn,
        "false_alarms": fp
    }

results = pd.DataFrame([
    evaluate("RF - Week 6", make_model(*candidates["RF balanced"])),
    evaluate("RF - tuned", gs_recall.best_estimator_),
    evaluate("SVC - Week 6", make_model(*candidates["SVC balanced"])),
    evaluate("SVC - tuned", rs_svc.best_estimator_)
]).sort_values("cv_recall", ascending=False)

results["clears_target"] = (
    (results["test_recall"] >= TARGET_RECALL) &
    (results["test_f1"] >= GUARD_F1)
)

print(results.to_string(index=False))

       model  cv_recall  cv_std  cv_f1  test_recall  test_f1  found  missed  false_alarms  clears_target
 SVC - tuned      0.766   0.038  0.557        0.673    0.507     37      18            54          False
  RF - tuned      0.739   0.041  0.541        0.745    0.536     41      14            57           True
SVC - Week 6      0.721   0.046  0.559        0.618    0.496     34      21            48          False
 RF - Week 6      0.656   0.054  0.523        0.655    0.545     36      19            41          False


write the recommendation

RF – Tuned Model
1. Model name and settings
- Model: Random Forest (RF – tuned)
- Settings:class_weight': balanced, max_depth:3, min_samples_leaf:1
2. Evidence
- CV Recall: 0.739 (73.9%)
- CV Standard Deviation: 0.041
- CV F1-score: 0.541
- Test Recall: 0.745 (74.5%)
- Test F1-score: 0.536
- Clears Target: True
3. What it beats
- Achieves the highest test recall (74.5%) among the four models.
- Finds more patients and misses fewer patients than the other models.
- Has a lower test F1-score than RF – Week 6 (0.545).
4. Patient findings
- Found: 41 patients
- Missed: 14 patients
- False alarms: 57 patients

save the final settings

In [18]:
final_settings = {
    "model": "RandomForestClassifier",
    "params": gs_recall.best_params_,
    "random_state": 42,
    "threshold": 0.5,
    "features": list(x.columns),
    "split": "test_size=0.2, random_state=42, stratify=y",
}

for k, v in final_settings.items():
    print(f"{k:13s} {v}")

results.to_csv(r"C:\Users\bachu\Desktop\5thsem_OJT\project1_diabetes\reportsproject1_diabetes_raw_p1W7v2_results.csv", index=False)
print("\nSaved: project1_diabetes_raw_p1W7v2_results.csv")

model         RandomForestClassifier
params        {'class_weight': 'balanced', 'max_depth': 3, 'min_samples_leaf': 1}
random_state  42
threshold     0.5
features      ['age', 'pregnancies', 'glucose', 'blood_pressure', 'skin_thickness', 'insulin', 'bmi', 'diabetes_pedigree', 'bmi_category_obese', 'bmi_category_overweight', 'bmi_category_underweight', 'glucose_band_normal', 'glucose_band_prediabetic']
split         test_size=0.2, random_state=42, stratify=y

Saved: project1_diabetes_raw_p1W7v2_results.csv
